# XGBoost LambdaMART LTR Model - Complete watsonx.ai Flow

This notebook provides a complete end-to-end pipeline for:
1. Training XGBoost LambdaMART model with optimized parameters
2. Local testing and validation
3. Saving model artifacts
4. Deploying to watsonx.ai deployment space

## Model Information
- **Algorithm**: XGBoost LambdaMART
- **Objective**: rank:ndcg
- **Deployment Type**: xgboost_2.0
- **Python Version**: 3.11
- **Key Features**:
  - Lower learning rate (0.05) for better convergence
  - Increased regularization (L1=0.15, L2=0.15) for stability
  - Level-wise tree growth for more stable training
  - Early stopping to prevent overfitting
  - Categorical feature support enabled

## IBM watsonx.ai Python SDK Documentation
**https://ibm.github.io/watsonx-ai-python-sdk/**

## Table of Contents

1. Setup and Configuration
2. Load and Preprocess Data
3. Feature Engineering
4. Prepare Training Data
5. Train XGBoost LambdaMART Model
6. Evaluate Model Performance
7. Test Prediction Function
8. Save Model Artifacts
9. Connect to watsonx.ai
10. Store Model in watsonx.ai
11. Deploy Model to Deployment Space
12. Get Deployment Details
13. Test Deployment
14. Summary and Next Steps

## 1. Setup and Configuration

Required packages:
  - xgboost==2.0.3
  - numpy==1.26.4
  - pandas==2.1.4
  - scikit-learn==1.3.0
  - ibm-watsonx-ai==1.5.3
  - ibm-cos-sdk==2.14.2

**These packages are pre-installed by default in the Python 3.11 Runtime 24.1.**

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import pickle
import tarfile
import json
import os
from datetime import datetime, timezone, timedelta
from importlib.metadata import version
import warnings
warnings.filterwarnings('ignore')

# Set timezone to UTC+8 (Asia/Taipei)
TZ_UTC8 = timezone(timedelta(hours=8))

# Get package versions
xgb_version = version('xgboost')
np_version = version('numpy')
pd_version = version('pandas')
sklearn_version = version('scikit-learn')
watsonx_version = version('ibm-watsonx-ai')
cos_version = version('ibm-cos-sdk')

print("="*80)
print("PACKAGE VERSIONS")
print("="*80)
print(f"✅ XGBoost version: {xgb_version}")
print(f"✅ NumPy version: {np_version}")
print(f"✅ Pandas version: {pd_version}")
print(f"✅ Scikit-learn version: {sklearn_version}")
print(f"✅ IBM watsonx.ai version: {watsonx_version}")
print(f"✅ IBM COS SDK version: {cos_version}")
print("\n✅ All libraries imported successfully")

PACKAGE VERSIONS
✅ XGBoost version: 2.0.3
✅ NumPy version: 1.26.4
✅ Pandas version: 2.1.4
✅ Scikit-learn version: 1.3.0
✅ IBM watsonx.ai version: 1.5.3
✅ IBM COS SDK version: 2.14.2

✅ All libraries imported successfully


## 2. Load and Preprocess Data

### Utilize the "Code Snippets" feature on Jupyter Notebook Editor to generate a code snippet to load data from a data asset or connection into your notebook!

In [ ]:
# Insert the code snippet here!!! ⤵️
#
# import os, types
# import pandas as pd
# from botocore.client import Config
# import ibm_boto3
# 
# ...
# 
# 
# 

#### Example:

In [ ]:
# import os, types
# import pandas as pd
# from botocore.client import Config
# import ibm_boto3

# def __iter__(self): return 0

# # @hidden_cell
# # The following code accesses a file in your IBM Cloud Object Storage. It includes your credentials.
# # You might want to remove those credentials before you share the notebook.

# cos_client = ibm_boto3.client(service_name='s3',
#     ibm_api_key_id='auto-generated_ibm_api_key_id',
#     ibm_auth_endpoint="https://iam.cloud.ibm.com/identity/token",
#     config=Config(signature_version='oauth'),
#     endpoint_url='https://s3.direct.us-south.cloud-object-storage.appdomain.cloud')

# bucket = 'bucket_name'
# object_key = 'LTR_model_input.csv'

# body = cos_client.get_object(Bucket=bucket,Key=object_key)['Body']
# # add missing __iter__ method, so pandas accepts body as file-like object
# if not hasattr(body, "__iter__"): body.__iter__ = types.MethodType( __iter__, body )

# df_import_from_project_bucket = pd.read_csv(body)
# df_import_from_project_bucket.head(10)

,cust_id,product_id(has 9 different categories),share_feature_1,product_feature_2,"relevance_score(1,2,3)\n3 > successfully applied prodcut in recent month\n2 > in the applied process but not successfully applied in recent month\n1 > only browse prodcut info via bank app in recent month"
0,a01,pl,100,32,1
1,a01,hy,100,21,3
2,b01,pl,500,0,3
3,b01,hy,500,74,1
4,b01,ins,500,32,1


In [3]:
# Use the data loaded from IBM Cloud Object Storage
# Using deep=True to create a completely independent copy
# This ensures modifications to df won't affect df_1
df = df_import_from_project_bucket.copy(deep=True)   # Deep copy is set to True in default

print(f"✅ Data loaded from Cloud Object Storage: {df.shape}")
print(f"\nFirst few rows:")
display(df.head())

✅ Data loaded from Cloud Object Storage: (5, 5)

First few rows:


,cust_id,product_id(has 9 different categories),share_feature_1,product_feature_2,"relevance_score(1,2,3)\n3 > successfully applied prodcut in recent month\n2 > in the applied process but not successfully applied in recent month\n1 > only browse prodcut info via bank app in recent month"
0,a01,pl,100,32,1
1,a01,hy,100,21,3
2,b01,pl,500,0,3
3,b01,hy,500,74,1
4,b01,ins,500,32,1


### Clean column names

In [4]:
# Clean column names
df.columns = df.columns.str.strip()

# Rename columns for easier handling
column_mapping = {
    'cust_id': 'cust_id',
    'product_id(has 9 different categories)': 'product_id',
    'share_feature_1': 'share_feature_1',
    'product_feature_2': 'product_feature_2'
}

# Find the relevance score column
relevance_col = [col for col in df.columns if 'relevance_score' in col][0]
column_mapping[relevance_col] = 'relevance_score'

df = df.rename(columns=column_mapping)

print("✅ Columns cleaned:")
print(df.columns.tolist())
print(f"\nData shape: {df.shape}")

✅ Columns cleaned:
['cust_id', 'product_id', 'share_feature_1', 'product_feature_2', 'relevance_score']

Data shape: (5, 5)


## 3. Feature Engineering

In [5]:
# Encode product_id
product_encoder = LabelEncoder()
df['product_id_encoded'] = product_encoder.fit_transform(df['product_id'])

print("✅ Product ID Encoding:")
for product, code in zip(product_encoder.classes_, range(len(product_encoder.classes_))):
    print(f"   {product} -> {code}")

# Define feature columns (order matters!)
feature_columns = ['product_id_encoded', 'share_feature_1', 'product_feature_2']

print(f"\n✅ Feature columns: {feature_columns}")
print(f"\nData with encoded features:")
display(df[['cust_id', 'product_id', 'product_id_encoded', 'share_feature_1', 'product_feature_2', 'relevance_score']].head(10))

✅ Product ID Encoding:
   hy -> 0
   ins -> 1
   pl -> 2

✅ Feature columns: ['product_id_encoded', 'share_feature_1', 'product_feature_2']

Data with encoded features:


,cust_id,product_id,product_id_encoded,share_feature_1,product_feature_2,relevance_score
0,a01,pl,2,100,32,1
1,a01,hy,0,100,21,3
2,b01,pl,2,500,0,3
3,b01,hy,0,500,74,1
4,b01,ins,1,500,32,1


## 4. Prepare Training Data

In [6]:
# Sort by cust_id to ensure group integrity
df = df.sort_values('cust_id').reset_index(drop=True)

# Prepare features and labels
X = df[feature_columns].values
y = df['relevance_score'].values

# Calculate group sizes (number of products per customer)
group_sizes = df.groupby('cust_id').size().values

print(f"✅ Training data prepared:")
print(f"   Total samples: {len(X)}")
print(f"   Number of groups (customers): {len(group_sizes)}")
print(f"   Group sizes: {group_sizes}")
print(f"   Feature matrix shape: {X.shape}")
print(f"   Label vector shape: {y.shape}")

✅ Training data prepared:
   Total samples: 5
   Number of groups (customers): 2
   Group sizes: [2 3]
   Feature matrix shape: (5, 3)
   Label vector shape: (5,)


## 5. Train XGBoost LambdaMART Model

In [7]:
# XGBoost parameters for ranking (optimized for stability and performance)
params = {
    'objective': 'rank:ndcg',      # LambdaMART objective
    'eval_metric': 'ndcg',         # Evaluation metric
    'eta': 0.05,                   # ⭐ Lower learning rate for better convergence
    'max_depth': 6,                # Maximum tree depth
    'min_child_weight': 5,         # Minimum sum of instance weight
    'subsample': 0.8,              # Row sampling
    'colsample_bytree': 0.8,       # Column sampling
    'alpha': 0.15,                 # ⭐ Increased L1 regularization for stability
    'lambda': 0.15,                # ⭐ Increased L2 regularization for stability
    'tree_method': 'hist',         # Histogram-based algorithm
    'grow_policy': 'depthwise',    # ⭐ Level-wise (depth-wise) growth for stability
    'max_cat_to_onehot': 8,        # ⭐ Adjusted for better categorical handling
    'seed': 42
}

print("📋 Optimized Parameters:")
print(f"   🎯 Growth Strategy: Level-wise (depthwise) - More stable")
print(f"   🎯 Learning Rate: {params['eta']} - Lower for better convergence")
print(f"   🎯 Regularization: L1={params['alpha']}, L2={params['lambda']} - Increased")
print(f"   🎯 Categorical Support: max_cat_to_onehot={params['max_cat_to_onehot']}")
print(f"\n   Full parameters:")
for key, value in params.items():
    print(f"      {key}: {value}")

# Create DMatrix with group information and categorical feature support
print(f"\n🔧 Creating DMatrix with categorical feature support...")
dtrain = xgb.DMatrix(X, label=y, enable_categorical=True)
dtrain.set_group(group_sizes)
print(f"   ✅ Column 0 (product_id_encoded) marked as categorical")

# Train the model with early stopping
num_boost_round = 200  # ⭐ Increased iterations
early_stopping_rounds = 15  # ⭐ Early stopping

print(f"\n🚀 Training with up to {num_boost_round} boosting rounds...")
print(f"   ⭐ Early stopping enabled: stops if no improvement for {early_stopping_rounds} rounds\n")

model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtrain, 'train')],
    early_stopping_rounds=early_stopping_rounds,
    verbose_eval=10
)

print(f"\n   ✅ Training stopped at iteration: {model.best_iteration + 1}")
print(f"   ✅ Best NDCG score: {model.best_score:.6f}")
print("\n✅ Model training completed!")

📋 Optimized Parameters:
   🎯 Growth Strategy: Level-wise (depthwise) - More stable
   🎯 Learning Rate: 0.05 - Lower for better convergence
   🎯 Regularization: L1=0.15, L2=0.15 - Increased
   🎯 Categorical Support: max_cat_to_onehot=8

   Full parameters:
      objective: rank:ndcg
      eval_metric: ndcg
      eta: 0.05
      max_depth: 6
      min_child_weight: 5
      subsample: 0.8
      colsample_bytree: 0.8
      alpha: 0.15
      lambda: 0.15
      tree_method: hist
      grow_policy: depthwise
      max_cat_to_onehot: 8
      seed: 42

🔧 Creating DMatrix with categorical feature support...
   ✅ Column 0 (product_id_encoded) marked as categorical

🚀 Training with up to 200 boosting rounds...
   ⭐ Early stopping enabled: stops if no improvement for 15 rounds

[0]	train-ndcg:0.85490
[10]	train-ndcg:0.85490
[14]	train-ndcg:0.85490

   ✅ Training stopped at iteration: 1
   ✅ Best NDCG score: 0.854905

✅ Model training completed!


## 6. Evaluate Model Performance

In [8]:
# Make predictions
predictions = model.predict(dtrain)
df['predicted_score'] = predictions

print("📊 Predictions vs Actual:")
display(df[['cust_id', 'product_id', 'relevance_score', 'predicted_score']].head(10))

# Show ranking for each customer
print("\n" + "="*80)
print("RANKING RESULTS BY CUSTOMER")
print("="*80)

for cust_id in df['cust_id'].unique():
    cust_data = df[df['cust_id'] == cust_id].copy()
    cust_data = cust_data.sort_values('predicted_score', ascending=False)
    
    print(f"\n📍 Customer: {cust_id}")
    print("Rank | Product | Relevance | Predicted Score")
    print("-" * 50)
    for idx, (_, row) in enumerate(cust_data.iterrows(), 1):
        print(f"{idx:4d} | {row['product_id']:7s} | {row['relevance_score']:9.0f} | {row['predicted_score']:15.4f}")

📊 Predictions vs Actual:


,cust_id,product_id,relevance_score,predicted_score
0,a01,pl,1,-2.753282e-08
1,a01,hy,3,-2.753282e-08
2,b01,pl,3,-2.753282e-08
3,b01,hy,1,-2.753282e-08
4,b01,ins,1,-2.753282e-08



RANKING RESULTS BY CUSTOMER

📍 Customer: a01
Rank | Product | Relevance | Predicted Score
--------------------------------------------------
   1 | pl      |         1 |         -0.0000
   2 | hy      |         3 |         -0.0000

📍 Customer: b01
Rank | Product | Relevance | Predicted Score
--------------------------------------------------
   1 | pl      |         3 |         -0.0000
   2 | hy      |         1 |         -0.0000
   3 | ins     |         1 |         -0.0000


## 7. Test Prediction Function

In [10]:
def predict_ranking(model, test_data):
    """
    Predict ranking scores for products
    
    Parameters:
    -----------
    model : xgboost.Booster
        Trained XGBoost model
    test_data : list of lists or numpy array
        Each row: [product_id_encoded, share_feature_1, product_feature_2]
    
    Returns:
    --------
    numpy.ndarray : Predicted scores
    """
    X_test = np.array(test_data, dtype=np.float32)
    dtest = xgb.DMatrix(X_test)
    predictions = model.predict(dtest)
    return predictions

# Test case 1: Customer a01
print("🧪 Test Case 1: Customer a01")
test_data_1 = [
    [0, 100, 21],  # hy
    [2, 100, 32]   # pl
]
predictions_1 = predict_ranking(model, test_data_1)
print(f"   Input: {test_data_1}")
print(f"   Predictions: {predictions_1}")
print(f"   Ranking: hy={predictions_1[0]:.4f}, pl={predictions_1[1]:.4f}")

# Test case 2: Customer b01
print("\n🧪 Test Case 2: Customer b01")
test_data_2 = [
    [0, 500, 74],  # hy
    [2, 500, 0],   # pl
    [1, 500, 32]   # ins
]
predictions_2 = predict_ranking(model, test_data_2)
print(f"   Input: {test_data_2}")
print(f"   Predictions: {predictions_2}")
print(f"   Ranking: hy={predictions_2[0]:.4f}, pl={predictions_2[1]:.4f}, ins={predictions_2[2]:.4f}")

print("\n✅ Prediction function works correctly!")

🧪 Test Case 1: Customer a01
   Input: [[0, 100, 21], [2, 100, 32]]
   Predictions: [-2.7532824e-08 -2.7532824e-08]
   Ranking: hy=-0.0000, pl=-0.0000

🧪 Test Case 2: Customer b01
   Input: [[0, 500, 74], [2, 500, 0], [1, 500, 32]]
   Predictions: [-2.7532824e-08 -2.7532824e-08 -2.7532824e-08]
   Ranking: hy=-0.0000, pl=-0.0000, ins=-0.0000

✅ Prediction function works correctly!


## 8. Save Model Artifacts

In [11]:
# Generate timestamp for file naming
timestamp = datetime.now(TZ_UTC8).strftime("%Y%m%d_%H%M%S")

# Save XGBoost model
model_filename = f'ltr_xgboost_ranker_{timestamp}.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(model, f)
print(f"✅ Model saved: {model_filename}")

# Save encoders
encoders = {
    'product_id': product_encoder
}
encoders_filename = f'ltr_xgboost_encoders_{timestamp}.pkl'
with open(encoders_filename, 'wb') as f:
    pickle.dump(encoders, f)
print(f"✅ Encoders saved: {encoders_filename}")

# Save metadata
metadata = {
    'feature_columns': feature_columns,
    'product_encoding': dict(zip(product_encoder.classes_, range(len(product_encoder.classes_)))),
    'timestamp': timestamp,
    'xgboost_version': xgb.__version__,
    'model_params': params,
    'num_boost_round': num_boost_round,
    'deployment_type': 'xgboost_2.0',
    'python_version': '3.11'
}
metadata_filename = f'ltr_xgboost_metadata_{timestamp}.pkl'
with open(metadata_filename, 'wb') as f:
    pickle.dump(metadata, f)
print(f"✅ Metadata saved: {metadata_filename}")

# Create ZIP file
archive_filename = f'ltr_xgboost_ranker_{timestamp}.tar.gz'
with tarfile.open(archive_filename, 'w:gz') as tar:
    tar.add(model_filename, arcname=model_filename)
print(f"✅ Archive created: {archive_filename}")

print("\n" + "="*80)
print("MODEL ARTIFACTS SAVED")
print("="*80)
print(f"\n📦 Files created:")
print(f"   - Model: {model_filename}")
print(f"   - Archive: {archive_filename}")
print(f"   - Encoders: {encoders_filename}")
print(f"   - Metadata: {metadata_filename}")

✅ Model saved: ltr_xgboost_ranker_20260318_122440.pkl
✅ Encoders saved: ltr_xgboost_encoders_20260318_122440.pkl
✅ Metadata saved: ltr_xgboost_metadata_20260318_122440.pkl
✅ Archive created: ltr_xgboost_ranker_20260318_122440.tar.gz

MODEL ARTIFACTS SAVED

📦 Files created:
   - Model: ltr_xgboost_ranker_20260318_122440.pkl
   - Archive: ltr_xgboost_ranker_20260318_122440.tar.gz
   - Encoders: ltr_xgboost_encoders_20260318_122440.pkl
   - Metadata: ltr_xgboost_metadata_20260318_122440.pkl


## 9. Connect to watsonx.ai

### Configuration

**IMPORTANT**: Update these values with your watsonx.ai credentials **ACCORDING TO YOUR ENVIRONMENT**.

#### CP4D (on-prem)

In [ ]:
# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================

username = 'PASTE YOUR USERNAME HERE'
url = 'PASTE THE PLATFORM URL HERE'

print("✅ Configuration loaded")

In [ ]:
# CP4D (on-prem)
from ibm_watsonx_ai import Credentials, APIClient

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.1"
)

# Alternatively you can use username and password to authenticate WML services.
# 
# credentials = Credentials(
#     username=***,
#     password=***,
#     url=***,
#     instance_id="openshift",
#     version="5.1"
# )

client = APIClient(credentials)

print("✅ Connected to Watson Machine Learning")
print(f"   Version: {client.version}")

#### IBM Cloud

In [18]:
# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================

url = 'https://us-south.ml.cloud.ibm.com'

print("✅ Configuration loaded")

✅ Configuration loaded


In [19]:
# IBM Cloud
from ibm_watsonx_ai import Credentials, APIClient
import getpass

credentials = Credentials(
    url=url,
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

client = APIClient(credentials)

print("✅ Connected to Watson Machine Learning")
print(f"   Version: {client.version}")

Enter your watsonx.ai api key and hit enter:  ········


✅ Connected to Watson Machine Learning
   Version: 1.5.3


### Set default space

In [20]:
# Project and Space IDs (get from watsonx.ai)

# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================

PROJECT_ID = "2c350ecf-e506-4409-a0a3-b7c2b8cba701"  # Replace with your project ID
SPACE_ID = "165e735f-11f6-4879-a6fd-99cfecf44f37"  # Replace with your deployment space ID

print("✅ Configuration loaded")
print(f"   PROJECT_ID: {PROJECT_ID}")
print(f"   SPACE_ID: {SPACE_ID}")

✅ Configuration loaded
   PROJECT_ID: 2c350ecf-e506-4409-a0a3-b7c2b8cba701
   SPACE_ID: 165e735f-11f6-4879-a6fd-99cfecf44f37


In [21]:
# Set default space
client.set.default_space(SPACE_ID)

print(f"✅ Default space set to: {SPACE_ID}")
print(f"\nSpace details:")
space_details = client.spaces.get_details(SPACE_ID)
print(f"   Name: {space_details['entity']['name']}")

✅ Default space set to: 165e735f-11f6-4879-a6fd-99cfecf44f37

Space details:
   Name: taishin_ai_gov_enablement_and_poc_dev_ds_dallas


## 10. Store Model in watsonx.ai

In [22]:
# List model files in current directory
import os
import glob

print("="*80)
print("MODEL FILES IN CURRENT DIRECTORY")
print("="*80)

# List all .pkl files
pkl_files = glob.glob('*.pkl')
if pkl_files:
    print(f"\n📦 Found {len(pkl_files)} .pkl file(s):")
    for f in sorted(pkl_files):
        size = os.path.getsize(f) / 1024  # Size in KB
        print(f"   - {f} ({size:.2f} KB)")
else:
    print("\n⚠️  No .pkl files found")

# List all .tar.gz files
targz_files = glob.glob('*.tar.gz')
if targz_files:
    print(f"\n📦 Found {len(targz_files)} .tar.gz file(s):")
    for f in sorted(targz_files):
        size = os.path.getsize(f) / 1024  # Size in KB
        print(f"   - {f} ({size:.2f} KB)")
else:
    print("\n⚠️  No .tar.gz files found")

# Show the model file that will be used for deployment
print(f"\n🎯 Archive file to be deployed:")
print(f"   {archive_filename}")

# Verify file exists
if os.path.exists(archive_filename):
    print(f"   ✅ Archive exists and ready for deployment")
else:
    print(f"   ❌ Archive not found! Please run the training cells first.")

MODEL FILES IN CURRENT DIRECTORY

📦 Found 3 .pkl file(s):
   - ltr_xgboost_encoders_20260318_122440.pkl (0.27 KB)
   - ltr_xgboost_metadata_20260318_122440.pkl (0.54 KB)
   - ltr_xgboost_ranker_20260318_122440.pkl (14.69 KB)

📦 Found 1 .tar.gz file(s):
   - ltr_xgboost_ranker_20260318_122440.tar.gz (1.72 KB)

🎯 Archive file to be deployed:
   ltr_xgboost_ranker_20260318_122440.tar.gz
   ✅ Archive exists and ready for deployment


In [23]:
# Define model metadata

MODEL_NAME = model_filename

model_meta_props = {
    client.repository.ModelMetaNames.NAME: MODEL_NAME,
    client.repository.ModelMetaNames.TYPE: "xgboost_2.0",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_UID: client.software_specifications.get_id_by_name("runtime-24.1-py3.11")
}

print("📋 Model metadata:")
print(f"   Model Name: {MODEL_NAME}")
print(f"   Type: xgboost_2.0")
print(f"   Software Spec: runtime-24.1-py3.11")

📋 Model metadata:
   Model Name: ltr_xgboost_ranker_20260318_122440.pkl
   Type: xgboost_2.0
   Software Spec: runtime-24.1-py3.11


In [24]:
# Store model (using tar.gz archive)
print(f"\n🚀 Storing model in watsonx.ai...")
print(f"   Using archive: {archive_filename}")

published_model = client.repository.store_model(
    model=archive_filename,
    meta_props=model_meta_props
)

model_uid = client.repository.get_model_id(published_model)

print(f"\n✅ Model stored successfully!")
print(f"   Model UID: {model_uid}")


🚀 Storing model in watsonx.ai...
   Using archive: ltr_xgboost_ranker_20260318_122440.tar.gz

✅ Model stored successfully!
   Model UID: 60fe5ba1-eccd-46e1-949f-e55be6ba7f71


## 11. Deploy Model to Deployment Space

In [25]:
# Define deployment metadata

DEPLOYMENT_NAME = model_filename + "_deployment"

deployment_meta_props = {
    client.deployments.ConfigurationMetaNames.NAME: DEPLOYMENT_NAME,
    client.deployments.ConfigurationMetaNames.ONLINE: {}
}

print("📋 Deployment metadata:")
print(f"   Deployment Name: {DEPLOYMENT_NAME}")
print(f"   Type: Online")

📋 Deployment metadata:
   Deployment Name: ltr_xgboost_ranker_20260318_122440.pkl_deployment
   Type: Online


In [26]:
# Create deployment
print(f"\n🚀 Creating deployment...")

deployment = client.deployments.create(
    artifact_uid=model_uid,
    meta_props=deployment_meta_props
)

deployment_uid = client.deployments.get_id(deployment)

print(f"\n✅ Deployment created successfully!")
print(f"   Deployment UID: {deployment_uid}")
print(f"\n⏳ Waiting for deployment to be ready...")

# Wait for deployment to be ready
import time
max_wait = 300  # 5 minutes
wait_interval = 10  # 10 seconds
elapsed = 0

while elapsed < max_wait:
    deployment_details = client.deployments.get_details(deployment_uid)
    status = deployment_details['entity']['status']['state']
    
    if status == 'ready':
        print(f"\n✅ Deployment is ready!")
        break
    elif status == 'failed':
        print(f"\n❌ Deployment failed!")
        print(f"   Status: {deployment_details['entity']['status']}")
        break
    else:
        print(f"   Status: {status} (waiting {elapsed}s/{max_wait}s)")
        time.sleep(wait_interval)
        elapsed += wait_interval

if elapsed >= max_wait:
    print(f"\n⚠️  Deployment timeout after {max_wait}s")
    print(f"   Check deployment status manually in watsonx.ai")


🚀 Creating deployment...


######################################################################################

Synchronous deployment creation for id: '60fe5ba1-eccd-46e1-949f-e55be6ba7f71' started

######################################################################################


initializing
Note: Scikit-learn 1.3/xgboost 2.0 framework with runtime-24.1-py3.11 for Watson Machine Learning is deprecated and will be removed in future. Use Scikit-learn 1.6/XGBoost 2.1 with runtime-25.1-py3.12 instead. For details, see https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/pm_service_supported_frameworks.html
...................
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='57b88863-1ae3-4ece-a1f4-e1e36ee932e8'
-----------------------------------------------------------------------------------------------



✅ Deployment created successfully!
   Deployme

## 12. Get Deployment Details

In [28]:
# Get deployment details
deployment_details = client.deployments.get_details(deployment_uid)

print("")
print("="*80)
print("DEPLOYMENT INFORMATION")
print("="*80)
print(f"\n📋 Deployment Details:")
print(f"   Name: {deployment_details['metadata']['name']}")
print(f"   Status: {deployment_details['entity']['status']['state']}")
print(f"   Deployment UID: {deployment_uid}")
print(f"   Model UID: {model_uid}")

# Get scoring endpoint
scoring_endpoint = client.deployments.get_scoring_href(deployment_details)
print(f"\n🔗 Scoring Endpoint:")
print(f"   {scoring_endpoint}")

Note: Scikit-learn 1.3/xgboost 2.0 framework with runtime-24.1-py3.11 for Watson Machine Learning is deprecated and will be removed in future. Use Scikit-learn 1.6/XGBoost 2.1 with runtime-25.1-py3.12 instead. For details, see https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/pm_service_supported_frameworks.html

DEPLOYMENT INFORMATION

📋 Deployment Details:
   Name: ltr_xgboost_ranker_20260318_122440.pkl_deployment
   Status: ready
   Deployment UID: 57b88863-1ae3-4ece-a1f4-e1e36ee932e8
   Model UID: 60fe5ba1-eccd-46e1-949f-e55be6ba7f71

🔗 Scoring Endpoint:
   https://us-south.ml.cloud.ibm.com/ml/v4/deployments/57b88863-1ae3-4ece-a1f4-e1e36ee932e8/predictions


## 13. Test Deployment

In [29]:
# Prepare test payload
test_payload = {
    "input_data": [{
        "values": [
            [0, 100, 21],  # hy
            [2, 100, 32],  # pl
            [1, 100, 15]   # ins
        ]
    }]
}

print("🧪 Testing deployment with sample data...")
print(f"\nInput:")
print(test_payload)

# Make prediction
predictions = client.deployments.score(deployment_uid, test_payload)

print(f"\n✅ Prediction successful!")
print(f"\nOutput:")
print(predictions)

# Parse and display results
if 'predictions' in predictions:
    pred_values = predictions['predictions'][0]['values']
    print(f"\n📊 Predicted Scores:")
    
    # Each value is a list [prediction, probability], we want the first element
    if isinstance(pred_values[0], list):
        print(f"   hy (0):  {pred_values[0][0]:.6f}")
        print(f"   pl (2):  {pred_values[1][0]:.6f}")
        print(f"   ins (1): {pred_values[2][0]:.6f}")
    else:
        # If it's a single value
        print(f"   hy (0):  {pred_values[0]:.6f}")
        print(f"   pl (2):  {pred_values[1]:.6f}")
        print(f"   ins (1): {pred_values[2]:.6f}")
    
    print(f"\n🎯 Ranking (higher score = higher rank):")
    # Create ranking based on scores
    if isinstance(pred_values[0], list):
        scores = [(pred_values[0][0], 'hy'), (pred_values[1][0], 'pl'), (pred_values[2][0], 'ins')]
    else:
        scores = [(pred_values[0], 'hy'), (pred_values[1], 'pl'), (pred_values[2], 'ins')]
    
    scores.sort(reverse=True)
    for rank, (score, product) in enumerate(scores, 1):
        print(f"   {rank}. {product}: {score:.6f}")

🧪 Testing deployment with sample data...

Input:
{'input_data': [{'values': [[0, 100, 21], [2, 100, 32], [1, 100, 15]]}]}

✅ Prediction successful!

Output:
{'predictions': [{'fields': ['prediction', 'probability'], 'values': [[-2.75328240206818e-08, -2.75328240206818e-08], [-2.75328240206818e-08, -2.75328240206818e-08], [-2.75328240206818e-08, -2.75328240206818e-08]]}]}

📊 Predicted Scores:
   hy (0):  -0.000000
   pl (2):  -0.000000
   ins (1): -0.000000

🎯 Ranking (higher score = higher rank):
   1. pl: -0.000000
   2. ins: -0.000000
   3. hy: -0.000000


## 14. Summary and Next Steps

In [30]:
print("="*80)
print("✅ PIPELINE COMPLETED SUCCESSFULLY!")
print("="*80)

print(f"\n📊 Summary:")
print(f"   ✅ Model trained: {MODEL_NAME}")
print(f"   ✅ Model stored in watsonx.ai")
print(f"   ✅ Deployment created: {DEPLOYMENT_NAME}")
print(f"   ✅ Deployment tested successfully")

print(f"\n📋 Deployment Information:")
print(f"   Model UID: {model_uid}")
print(f"   Deployment UID: {deployment_uid}")
print(f"   Scoring Endpoint: {scoring_endpoint}")

print(f"\n📦 Local Files:")
print(f"   Model: {model_filename}")
print(f"   Archive: {archive_filename}")
print(f"   Encoders: {encoders_filename}")
print(f"   Metadata: {metadata_filename}")

✅ PIPELINE COMPLETED SUCCESSFULLY!

📊 Summary:
   ✅ Model trained: ltr_xgboost_ranker_20260318_122440.pkl
   ✅ Model stored in watsonx.ai
   ✅ Deployment created: ltr_xgboost_ranker_20260318_122440.pkl_deployment
   ✅ Deployment tested successfully

📋 Deployment Information:
   Model UID: 60fe5ba1-eccd-46e1-949f-e55be6ba7f71
   Deployment UID: 57b88863-1ae3-4ece-a1f4-e1e36ee932e8
   Scoring Endpoint: https://us-south.ml.cloud.ibm.com/ml/v4/deployments/57b88863-1ae3-4ece-a1f4-e1e36ee932e8/predictions

📦 Local Files:
   Model: ltr_xgboost_ranker_20260318_122440.pkl
   Archive: ltr_xgboost_ranker_20260318_122440.tar.gz
   Encoders: ltr_xgboost_encoders_20260318_122440.pkl
   Metadata: ltr_xgboost_metadata_20260318_122440.pkl


## Appendix: Useful Functions

In [ ]:
# Function to list all deployments
def list_deployments():
    """List all deployments in the current space"""
    deployments = client.deployments.get_details()
    print("📋 All Deployments:")
    for deployment in deployments['resources']:
        print(f"   - {deployment['entity']['name']} ({deployment['metadata']['id']})")
        print(f"     Status: {deployment['entity']['status']['state']}")

# Function to delete a deployment
def delete_deployment(deployment_id):
    """Delete a deployment by ID"""
    client.deployments.delete(deployment_id)
    print(f"✅ Deployment {deployment_id} deleted")

# Function to update deployment
def update_deployment(deployment_id, new_model_id):
    """Update deployment with a new model"""
    meta_props = {
        client.deployments.ConfigurationMetaNames.ASSET: {
            "id": new_model_id
        }
    }
    client.deployments.update(deployment_id, meta_props)
    print(f"✅ Deployment {deployment_id} updated with model {new_model_id}")

print("✅ Utility functions defined")

---

## 🎉 Congratulations!

You have successfully:
1. ✅ Trained an XGBoost LambdaMART model
2. ✅ Tested the model locally
3. ✅ Stored the model in watsonx.ai
4. ✅ Deployed the model to a deployment space
5. ✅ Tested the deployment

Your LTR model deployment is now ready to use!

---

**Model Type:** XGBoost LambdaMART  
**Deployment Type:** xgboost_2.0  
**Python Version:** 3.11  
**Status:** ✅ Deployment Ready